# Časť 4, Téma 3: CPA útok pomocou ChipWhisperer Analyzer (HLAVNÁ ČASŤ)


---
POZNÁMKA: Toto laboratórne cvičenie odkazuje na niektoré (komerčné) školiace materiály na stránke [ChipWhisperer.io](https://www.ChipWhisperer.io). Lab môžete voľne spúšťať a používať v súlade s open-source licenciou (vrátane použitia vo vašich vlastných kurzoch, ak ich budete distribuovať podobným spôsobom), ale musíte zachovať informáciu o tomto pôvodnom umiestnení. Ak si chcete užiť plnohodnotný zážitok, zvážte účasť na našom školení.

---

**ZHRNUTIE:** *Minule sme si ukázali, ako sa dá použiť korelácia na získanie kľúča AES, a taktiež efektivitu takéhoto útoku. V tomto labe si tento útok zopakujeme pomocou nástroja ChipWhisperer Analyzer a získame o celom útoku dodatočné informácie.*

**CIELE VZDELÁVANIA:**

* Použiť ChipWhisperer Analyzer na vykonanie CPA útoku
* Vykresliť dodatočné informácie o priebehu útoku

## Požiadavky

Zadržte! Skôr než budete pokračovať, uistite sa, že máte prebraté nasledujúce návody:

* ☑ CPA on Firmware Implementation of AES (mali by ste rozumieť tomu, ako funguje CPA útok).
* ☑ SCA101 Intro (mali by ste mať predstavu o tom, ako spustiť verzie špecifické pre daný hardvér).

## Projekty

Tentokrát nebudeme potrebovať žiadne modely, implementácie SBoxov ani nič podobné. Namiesto toho je všetko integrované priamo v ChipWhisperer Analyzer. Ďalšou zmenou oproti predchádzajúcim návodom je, že namiesto numpy polí budeme používať projekty ChipWhisperer, keďže väčšina funkcií ChipWhisperer Analyzer pracuje výhradne s nimi.


In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEXMEGA'
CRYPTO_TARGET='AVRCRYPTOLIB' 
SS_VER='SS_VER_2_1'

In [ ]:
%run "../Setup_Scripts/Setup_Generic.ipynb"

In [ ]:
%%bash -s "$PLATFORM" "$CRYPTO_TARGET" "$SS_VER"
cd ../../firmware/mcu/simpleserial-aes
make PLATFORM=$1 CRYPTO_TARGET=$2 SS_VER=$3 -j

In [ ]:
cw.program_target(scope, prog, "../../firmware/mcu/simpleserial-aes/simpleserial-aes-{}.hex".format(PLATFORM))

In [ ]:
proj = cw.create_project("Lab 4_3", overwrite=True)

In [ ]:
from tqdm.notebook import trange
import numpy as np
import time

ktp = cw.ktp.Basic()
trace_array = []
textin_array = []

N = 50
for i in trange(N, desc='Capturing traces'):
    key, text = ktp.next()
    trace = cw.capture_trace(scope, target, text, key)
    if not trace:
        continue
    
    proj.traces.append(trace)

Skôr než prejdeme k samotnému CPA útoku, rýchlo sa pozrime na samotné projekty:

In [ ]:
# we can access wave, textin, etc as a whole with proj.traces
for trace in proj.traces:
    print(trace.wave[0], trace.textin, trace.textout, trace.key)

# can also access individually with proj.waves, proj.textins, etc.
for wave in proj.waves:
    print(wave[0])

## ChipWhisperer Analyzer

K modulu Analyzer môžeme pristupovať cez `chipwhisperer.analyzer`:

In [ ]:
import chipwhisperer.analyzer as cwa

Taktiež musíme nastaviť náš model úniku (leakage model) na výstup z SBoxu. ChipWhisperer Analyzer obsahuje množstvo rôznych modelov úniku, ktoré sú užitočné v najrôznejších situáciách. Detailnejšie sa na ne pozrieme v kurze SCA201.

In [ ]:
leak_model = cwa.leakage_models.sbox_output

Zvyšok konfigurácie zaberie iba jeden riadok:

In [ ]:
attack = cwa.cpa(proj, leak_model)

Ak si chcete prezrieť nastavenia útoku, môžete si objekt `cpa` jednoducho vytlačiť:

In [ ]:
print(attack)

Spustenie útoku sa taktiež vykonáva na jednom riadku:

In [ ]:
results = attack.run()

Poďme sa pozrieť, či sa nám podarilo získať kľúč AES:

In [ ]:
print(results)

Kompletné informácie o útoku môžeme získať zavolaním `results.find_maximums()`, čo vracia štruktúru:

```Python
find_maxiums() ->
    [subkey0_data, subkey1_data, subkey2_data, ...]
    
subkey0_data ->
    [guess0, guess1, guess2, ...]
    
guess0 ->
    (key_guess, location_of_max, correlation)
```

Ak by ste chceli napríklad vytlačiť koreláciu pre tretí najlepší odhad 4. subkľúča (subkey), spustili by ste:

```python
print(attack_results.find_maximums()[4][3][2])
```

Všimnite si, že hodnota "location of the max" (index vzorky s maximom) sa štandardne nepočíta/nesleduje, a preto vracia 0. Použitie knižnice pandas nám umožní zobraziť tieto dáta prehľadne v tabuľke DataFrame. Aby sme dosiahli očakávanú orientáciu, musíme dáta transponovať:

In [ ]:
import pandas as pd
stat_data = results.find_maximums()
df = pd.DataFrame(stat_data).transpose()
print(df.head())

Ešte lepšie je, že môžeme využiť metódu `.style` na pokročilejšie prispôsobenie vzhľadu. To nám umožňuje reťaziť formátovacie funkcie. Môžeme napríklad odstrániť prebytočné nuly a sprehľadniť dáta. Keďže poznáme správny kľúč, môžeme dokonca nechať správny kľúč vypísať inou farbou!

Vďaka knižnici pandas máte k dispozícii obrovské možnosti formátovania! Pre viac informácií si pozrite dokumentáciu na https://pandas.pydata.org/pandas-docs/stable/style.html.

In [ ]:
key = proj.keys[0]
def format_stat(stat):
    return str("{:02X}<br>{:.3f}".format(stat[0], stat[2]))

def color_corr_key(row):
    global key
    ret = [""] * 16
    for i,bnum in enumerate(row):
        if bnum[0] == key[i]:
            ret[i] = "color: red"
        else:
            ret[i] = ""
    return ret

df.head().style.format(format_stat).apply(color_corr_key, axis=1)

Na vrchu tabuľky by ste mali vidieť červeno sfarbené čísla. Gratulujeme, práve ste úspešne vykonali CPA útok proti AES!

Ďalej sa pozrieme na to, ako môžeme využiť ďalšie funkcie modulu Analyzer na zefektívnenie celého procesu útoku a na lepšiu interpretáciu získaných dát.

## Intervaly hlásenia (Reporting Intervals)

Keď sme spustili `attack.run()`, spracovali sa všetky výkonové krivky naraz predtým, než sme dostali akýkoľvek výstup. ChipWhisperer Analyzer však v skutočnosti využíva „online“ výpočet korelácie, ktorý sme spomínali minule, čo znamená, že môžeme získavať priebežné výsledky už počas behu útoku. To sa dá dosiahnuť vytvorením callback funkcie, ktorú odovzdáme ako argument metóde `attack.run()`. Táto funkcia sa zavolá zakaždým, keď útok prejde určeným intervalom aktualizácie (predvolená hodnota je 25, čo predstavuje druhý parameter pre `attack.run()`).

Využime to na aktualizáciu našej tabuľky po každých 10 krivkách. Väčšina práce spočíva len v presunutí nášho existujúceho kódu do callback funkcie. Taktiež musíme použiť funkciu `clear_output` na premazanie predchádzajúcej tabuľky a `display()`, aby sa nová tabuľka reálne vykreslila:

In [ ]:
from IPython.display import clear_output
import numpy as np
        
def stats_callback():
    results = attack.results
    results.set_known_key(key)
    stat_data = results.find_maximums()
    df = pd.DataFrame(stat_data).transpose()
    clear_output(wait=True)
    display(df.head().style.format(format_stat).apply(color_corr_key,axis=1))
    
results = attack.run(stats_callback, 10)

K dispozícii je aj predvolený jupyter callback – na spustenie celého útoku vám potom stačia tieto **tri riadky**!

In [ ]:
import chipwhisperer as cw
cb = cwa.get_jupyter_callback(attack)
results = attack.run(cb, 10)

V tomto prípade sme nastavili interval hlásenia na každých 10 kriviek. V závislosti od typu útoku a od toho, čo presne chcete zistiť, môžete zvoliť vyššie alebo nižšie hodnoty: vo všeobecnosti platí, že menej časté hlásenia urýchľujú výpočet, no častejšie aktualizácie vám umožnia ukončiť príliš dlhý útok predčasne, ak kľúč nájdete skôr. Častejšie hlásenia tiež zvyšujú rozlíšenie niektorých dát v grafoch (na ktoré sa pozrieme vzápätí).

## Vykresľovanie dát (Plot Data)

Analyzer obsahuje aj modul na vytváranie grafov, ktoré vám pomôžu interpretovať nazbierané dáta. Tieto grafy pracujú vždy s jedným subkľúčom naraz a vracajú dáta, ktoré môžeme vykresliť pomocou knižnice bokeh (alebo iného preferovaného grafického modulu). Začnime tým, že si inicializujeme triedu, ktorá zastrešuje všetky tieto výpočty:

In [ ]:
plot_data = cwa.analyzer_plots(results)

Najskôr sa pozrieme na modul „Output Vs. Time“ (Výstup v závislosti od času), ktorý nám umožní vykresliť koreláciu našich odhadov v časovej osi. To je nesmierne užitočné na presné lokalizovanie miest, kde sa vykonávajú operácie, na ktoré útočíme. Podobne ako v predchádzajúcich návodoch, na zobrazenie výsledných dát použijeme bokeh.

Metóda, ktorá nás zaujíma, je `get_plot_data(bnum)`. Tá vracia zoznam v tvare: `[xrange, correct_key, incorrect_key_data, incorrect_key_data]` pre zadanú pozíciu bajtu `bnum`. Metóda vracia dve sady dát pre nesprávne kľúče z toho dôvodu, že jedna obsahuje odhady kľúčov s hodnotou nižšou ako správny kľúč a druhá odhady s hodnotou vyššou.

Keďže budeme mať k dispozícii obrovské množstvo bodov, vykreslíme ich štandardným spôsobom, ale na konci výstup zdecimujeme (zredukujeme hustotu bodov):

In [ ]:
def byte_to_color(idx):
    return hv.Palette.colormaps['Category20'](idx/16.0)

import holoviews as hv
from holoviews.operation.datashader import datashade, shade, dynspread, rasterize
from holoviews.operation import decimate
import pandas as pd, numpy as np

a = []
b = []
hv.extension('bokeh')
for i in range(0, 16):
    data = plot_data.output_vs_time(i)
    a.append(np.array(data[1]))
    b.append(np.array(data[2]))
    b.append(np.array(data[3]))
    
pda = pd.DataFrame(a).transpose().rename(str, axis='columns')
pdb = pd.DataFrame(b).transpose().rename(str, axis='columns')
curve = hv.Curve(pdb['0'], "Sample").options(color='black')
for i in range(1, 16):
    curve *= hv.Curve(pdb[str(i)]).options(color='black')
    
for i in range(0, 16):
    curve *= hv.Curve(pda[str(i)]).options(color=byte_to_color(i))
decimate(curve.opts(width=900, height=600))

V grafe by ste mali vidieť niekoľko výrazných špičiek (píkov). Najväčšia z nich indikuje moment, kedy reálne prebieha vyhľadávanie v SBoxe (tie menšie zvyčajne predstavujú iné operácie AES, ktoré iba presúvajú dáta zo SBoxu v pamäti). Keďže pracujeme s absolútnymi hodnotami, uvidíte tam aj záporné hodnoty.

Tieto informácie sú užitočné v mnohých smeroch. Môžete si napríklad všimnúť, že prvých 16 špičiek tvoriacich operáciu SBox zaberá iba malú časť z celkovej dĺžky meranej krivky. Ak by sme niekedy potrebovali tento útok spustiť znova, mohli by sme snímať omnoho menší počet vzoriek, čím by sme analýzu výrazne urýchlili!

### PGE v závislosti od počtu kriviek (PGE vs. Traces)

Ďalším typom dát, ktoré si vizualizujeme, je graf čiastočnej entrópie odhadu (PGE – Partial Guessing Entropy) v závislosti od počtu použitých kriviek. Ako už bolo spomenuté, PGE vyjadruje, o koľko pozícií nižšie od vrcholu tabuľky sa nachádza skutočný subkľúč medzi všetkými našimi odhadmi. Ak má napríklad 7 nesprávnych odhadov subkľúča vyššiu koreláciu ako ten skutočný, hovoríme, že subkľúč má PGE rovnú 7.

Tento graf skvele ukazuje, koľko kriviek bolo reálne potrebných na úspešné prelomenie danej implementácie AES. Majte však na pamäti, že rozlíšenie tohto grafu striktne závisí od nastaveného intervalu hlásení (zroveň platí, že funkcia `attack_results.find_maximums()` musí byť volaná priamo v callback funkcii). V našom prípade máme interval nastavený na 10, takže rozlíšenie v grafe bude po 10 krivkách.

Táto metóda funguje podobne ako predchádzajúci graf – ako argument prijíma index bajtu `bnum` a vracia zoznam v tvare `[xrange, PGE]`. 

In [ ]:
ret = plot_data.pge_vs_trace(0)
curve = hv.Curve((ret[0],ret[1]), "Traces Used in Calculation", "Partial Guessing Entrop of Byte")
for bnum in range(1, 16):
    ret = plot_data.pge_vs_trace(bnum)
    curve *= hv.Curve((ret[0],ret[1])).opts(color=byte_to_color(bnum))
curve.opts(width=900, height=600)

Mali by ste vidieť niekoľko čiar, ktoré začínajú na vysokých hodnotách a následne prudko klesajú k nule. Môžete si všimnúť, že implementáciu AES sme úspešne prelomili bez toho, aby sme vôbec museli spotrebovať všetky nasnímané krivky.

Aj keď sa nám podarilo kľúč odhaliť s menším počtom kriviek, v praxi by sme ich počet pri snímaní radšej neznižovali. Nezabúdajte na to, že kým tu presný kľúč vopred poznáme, pri reálnom útoku ho nepoznáme, a preto sa musíme spoliehať výhradne na hodnotu korelácie, aby sme určili, kedy je kľúč definitívne prelomený. Náš nasledujúci graf nám pomôže lepšie zhodnotiť, nakoľko bezpečné a reálne je znižovať počet snímaných kriviek.

### Korelácia v závislosti od počtu kriviek (Correlation vs. Traces)

Posledným grafom, na ktorý sa pozrieme, je priebeh korelácie v závislosti od počtu kriviek. Podobne ako pri grafe PGE, aj tu je rozlíšenie určené intervalom hlásení (v našom prípade 10). Táto metóda vracia zoznam v tvare `[xrange, [data_for_kguess]]`, čo znamená, že musíme vykresliť každý jeden odhad pre každý subkľúč. Rovnako ako predtým, správny odhad subkľúča vykreslíme meniacou sa farbou a všetky ostatné (nesprávne) odhady ponecháme čierne.

Ako sami uvidíte, všetky odhady subkľúčov začínajú na pomerne vysokých hodnotách korelácie, no s pribúdajúcimi krivkami všetky okrem toho správneho rýchlo klesajú k nule. Ak by ste kľúč vopred nepoznali, v ktorom bode by ste si boli stopercentne istí, že odhad s najvyššou koreláciou je skutočne ten správny subkľúč?

Poďme pokračovať a vykreslime korelácie pre správny odhad a pre druhý najlepší odhad v poradí:

In [ ]:
a = []
b = []
for bnum in range(0, 16):
    data = plot_data.corr_vs_trace(bnum)
    best = [0] * len(data[1][0])
    for i in range(256):
        if i == key[bnum]:
            a.append(np.array(data[1][i]))
        else:
            if max(best) < max(data[1][i]): best = data[1][i]
    b.append(np.array(best))

pda = pd.DataFrame(a).transpose().rename(str, axis='columns')
pdb = pd.DataFrame(b).transpose().rename(str, axis='columns')
curve = hv.Curve(pdb['0'].tolist(), "Iteration Number", "Max Correlation").options(color='black')
for i in range(1,len(pdb.columns)):
    curve *= hv.Curve(pdb[str(i)]).options(color='black')
    
for i in range(len(pda.columns)):
    curve *= hv.Curve(pda[str(i)]).options(color=byte_to_color(i))
            
curve.opts(width=900, height=600)

## Závery a ďalšie kroky

Ako ste mali možnosť vidieť, Analyzer robí spúšťanie CPA útokov omnoho jednoduchším v porovnaní s naším pôvodným manuálnym prístupom. Veľkou výhodou je tiež automatické logovanie zaujímavých dát a priebežné informovanie o úspešnosti útoku v reálnom čase.


---
<small>UPOZORNENIE: Tento materiál je chránený autorskými právami (C) NewAE Technology Inc., 2015–2020. ChipWhisperer je ochranná známka spoločnosti NewAE Technology Inc., na ktorú si táto spoločnosť uplatňuje práva vo všetkých jurisdikciách a ktorá je registrovaná minimálne v Spojených štátoch amerických, Európskej únii a Čínskej ľudovej republike.

Návody odvodené z našej open-source práce musia byť uverejnené pod príslušnou open-source licenciou a upozornenie na zdroj musí byť *jasne zobrazené*. Iba pôvodní držitelia autorských práv môžu udeľovať licencie alebo povoľovať ďalšiu distribúciu – hoci spoločnosť NewAE Technology Inc. vlastní autorské práva k mnohým návodom, repozitár github obsahuje príspevky komunity, ktoré nemôže licencovať za špeciálnych podmienok a **musia** byť udržiavané ako open-source verzia. V prípade potreby prosím kontaktujte NewAE Technology Inc ohľadom špeciálnych povolení (ak je to možné)

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.</small>